In [9]:


BEGIN TRY
    DECLARE @RunDate DATE;

    -- Get the current run date from audit table
    SELECT TOP 1
        @RunDate = CONVERT(DATE, process_date, 120)
    FROM [Dw_Bi_Audit].[dbo].[audit_file_run]
       WHERE silver_gold_id_status = 'In Progress'
      AND PipelineName = 'Pl_Product'
    ORDER BY process_date DESC;

    ----------- Check if RunDate is NULL -----------
    IF @RunDate IS NULL
    BEGIN
        THROW 50001, 'RunDate is NULL. Cannot proceed with deletion.', 1;
    END

    -- Delete Current data for RunDate if exists
    DELETE FROM [Dw_Bi_Reporting].[dbo].[Sales_Orders_EXCUTED]
    WHERE CAST([Exdate] AS DATE) = @RunDate;

    -- Optional: check run date
    --SELECT @Exdate AS RunDate_Used;

END TRY
BEGIN CATCH
    SELECT
        ERROR_MESSAGE()  AS ERROR_MESSAGE,
        ERROR_NUMBER()   AS ERROR_NUMBER,
        ERROR_SEVERITY() AS ERROR_SEVERITY;
    THROW;
END CATCH;


In [7]:
BEGIN TRY
    INSERT INTO [Dw_Bi_Reporting].[dbo].[Sales_Orders_EXCUTED]
    SELECT  
        [OrderID],
        [OrderDate],
        [CustomerID],
        [CustomerName],
        [ProductCategory],
        [ProductName],
        [Quantity],
        [UnitPrice],
        [Region],
        [SalesChannel],
        [SalesAmount],
        [OrderYear],
        [OrderMonth],
        [OrderValueBucket],
        [Id],
        [Exdate]
    FROM [lhs_product].[dbo].[sales_orders_exc_transformed]

    -- Old logic: load only today's data
     WHERE CAST([Exdate] AS DATE) = CAST(CURRENT_TIMESTAMP AS DATE);

    -- New logic: load data based on audit RunDate
    --WHERE CAST([Exdate] AS DATE) = @RunDate;

END TRY
BEGIN CATCH
    SELECT
        ERROR_MESSAGE() AS ERROR_MESSAGE,
        ERROR_NUMBER() AS ERROR_NUMBER,
        ERROR_SEVERITY() AS ERROR_SEVERITY;
    THROW;
END CATCH;


## Check for data loaded in Table

In [11]:


DECLARE @RunDate DATE;
DECLARE @CNT NUMERIC;

SELECT TOP 1
    @RunDate = CONVERT(Date, process_date, 120)
FROM [Dw_Bi_Audit].[dbo].[audit_file_run]
WHERE silver_gold_id_status = 'In Progress'
  AND PipelineName = 'Pl_Product'
order by process_date desc;

SELECT @CNT = COUNT(0)
FROM [Dw_Bi_Reporting].[dbo].[Sales_Orders_EXCUTED]
WHERE CAST([Exdate] AS DATE) = @RunDate;

--SELECT @CNT;
------------- Check if CNT is NULL -------------
IF @CNT = '0'
BEGIN
    THROW 50001, 'Data not loaded.', 1;
END
